In [1]:
from ultralytics import YOLO
import cv2
import numpy as np
import pytesseract
import os
import shutil
import pandas as pd
from datetime import datetime, timedelta

In [2]:
base_path = './IMG/plate_img'
new_path = base_path
i = 0
# Find a new directory name that doesn't exist yet
while os.path.exists(new_path):
    i += 1
    new_path = f"{base_path}{i}"
# Create the new directory
os.makedirs(new_path, exist_ok=True)

print(f"Save to {new_path}")

Save to ./IMG/plate_img44


In [ ]:
# Open the video capture
cap = cv2.VideoCapture('./testing_video/test6.mp4')

# Load the models
car_model = YOLO('car.pt')
# plate_model = YOLO('plate.pt')
plate_model = YOLO('./Results_yolov8s/yolov8s_yaml(3)/weights/best.pt')
plate_letter_model = YOLO('plate_letter.pt')

n_frame = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = 7 #68 31 (fps=7 second=1.5)
frame_count  = 0 

# Initialize variables
last_capture_time = datetime.min

count_box = 0
final_plate_letter = None
final_plate_box = 0
final_plate_letter_num = 0
final_count = 0
final_score = 0

df = None
fileName = None
num_count = []
file = []
num_plate = []
plate_capture = []
num_plate_letter = []
time = []
detect_score = []

# Define the output video file path
output_video_path = f'{new_path}/output_video.mp4'
# Define the codec and create VideoWriter object
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, 30, (frame_width, frame_height))

# Read and process each frame
while True:
    ret, frame = cap.read()  # Read a frame
    frame_count =  frame_count + 1
    if not ret:
        break  # Break the loop if there are no more frames

    classes_car_id = [2]
    plate_box = 0
    cleaned_letter = None
    plate_character_box = 0
    plate_character_OCR = []
    current_detection = []
    plate_character_YOLO = []
    plate_score = 0
    
    # Detect cars
    car_results = car_model(frame, conf=0.5)[0]
    for car_result in car_results.boxes.data.tolist():
        x1, y1, x2, y2, score, class_id = car_result
        if int(class_id) in classes_car_id:
            cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)  # Draw a green bounding box for cars
    
    # Detect plates
    plate_results = plate_model(frame, conf=0.5)[0]
    for plate_result in plate_results.boxes.data.tolist():
        x1, y1, x2, y2, score, class_id = plate_result

        # Draw bounding box
        cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 0, 255), 2)

        if frame_count % fps == 0:
            # Check if it's time to capture a new plate
            current_time = datetime.now()
            if current_time - last_capture_time >= timedelta(seconds=1.5):
                last_capture_time = current_time
                if (score >= 0.75):
                    # count_box += 1
                    # plate_box += 1
                    plate_score = score
                    fileName = '/plate'+str(count_box)+'.jpg'
                    plate_region = frame[int(y1):int(y2), int(x1):int(x2)]
                    gray_plate = cv2.cvtColor(plate_region, cv2.COLOR_BGR2GRAY) 
                    resized_plate_region = cv2.resize(gray_plate, (800, int(gray_plate.shape[0] * (800 / gray_plate.shape[1]))), interpolation=cv2.INTER_CUBIC)
    
                    # Define four points in the original image
                    pts_src = np.float32([[141, 131], [480, 159], [493, 630], [64, 601]])
                    # Define four points in the target image, which can be adjusted to the target position as needed
                    pts_dst = np.float32([[70, 40], [340, 120], [320, 400], [20, 235]])
                    # Calculate the perspective transformation moment 0
                    matrix = cv2.getPerspectiveTransform(pts_src, pts_dst)
                    # Applying Perspective Transformations
                    letter = cv2.warpPerspective(resized_plate_region, matrix, (630, 310))
                    # Perform bitwise inversion
                    invert = cv2.bitwise_not(letter)
                    # Perform OCR on the inverted image
                    text_letter = pytesseract.image_to_string(invert, lang='eng', config='--psm 6')
                    cleaned_letter = ''.join(e for e in text_letter if e.isalnum() and e.isdigit() or e.isupper())
                    plate_character_OCR = list(cleaned_letter)
                    # print(plate_character_OCR)
    
                    # Detect plate letter
                    letter_rgb = cv2.cvtColor(letter, cv2.COLOR_GRAY2RGB)
                    plate_letter_results = plate_letter_model.predict(letter_rgb, conf=0.5)[0]
                    for plate_letter_result in plate_letter_results.boxes.data.tolist():
                        x1, y1, x2, y2, score, class_id = plate_letter_result
                        plate_character_box += 1
                        # Draw bounding box
                        crop_character = cv2.rectangle(letter, (int(x1), int(y1)), (int(x2), int(y2)), (255, 0, 0), 2)
                        # Collect detected character labels
                        plate_character_YOLO.append(plate_letter_model.names[int(class_id)])
                        # print(plate_character_YOLO)
    
                    # Compare the sorted lists
                    if sorted(plate_character_OCR) == sorted(plate_character_YOLO) and plate_character_box != 0:
                        plate_box += 1
                        count_box += 1
                        
                        final_count = count_box
                        num_count.append(final_count)                  
                        final_plate_box = plate_box
                        num_plate.append(final_plate_box)              
                        final_plate_letter = cleaned_letter
                        plate_capture.append(final_plate_letter)                   
                        final_plate_letter_num = plate_character_box
                        num_plate_letter.append(final_plate_letter_num)
                        now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                        time.append(now)
                        final_score = plate_score
                        detect_score.append(final_score)
                        file.append(fileName)
                        
                        cv2.imwrite(new_path + fileName, plate_region, [cv2.IMWRITE_JPEG_QUALITY, 100])
                        combined_image = cv2.hconcat([plate_region, plate_region])
                        # Display the combined image in the "Plate" window (verify window name)
                        cv2.imshow("Plate", combined_image)
                        cv2.imshow("Plate box", crop_character)
                        # print("Plate:", ''.join(final_plate_letter))
                    else:
                        print("Plate result have different !!!")

    # Outprint the text on video
    cv2.putText(frame, f'Count: {final_count}', (20, 60), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2, cv2.LINE_AA)
    cv2.putText(frame, f'Plate: {final_plate_box}', (20, 95), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2, cv2.LINE_AA)
    cv2.putText(frame, f'Plate Letter: {final_plate_letter}', (20, 130), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2, cv2.LINE_AA)
    cv2.putText(frame, f'Plate Character: {final_plate_letter_num}', (20, 165), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2, cv2.LINE_AA)
    cv2.putText(frame, f'Score: {final_score:.2f}', (20, 195), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2, cv2.LINE_AA)
    # print(f'Plate: {final_plate_box}')

    # overwrite the original df
    df = pd.DataFrame({
        'File': file,
        'Plate': num_plate,
        'Plate_Letter': plate_capture,
        'Num_Plate': num_plate_letter,
        'Time': time,
        'Score': detect_score
    }, index=num_count)

    # Save the dataframe to a CSV file
    try:
        cvs_file = f"{new_path}/traindata.csv"
        if os.path.exists(cvs_file):
            os.remove(cvs_file)
        df.to_csv(cvs_file, index=True)
        print("traindata.csv created successfully.")
    except Exception as e:
        print(f"Error while saving traindata.csv: {e}")
        
    # Show the frame with bounding boxes
    cv2.imshow('Detected Objects', frame)
    out.write(frame)  # Write the frame to the output video
    # Check for key press
    key = cv2.waitKey(1)
    if key == ord('q'):  # Press 'q' to exit
        break

# Release the video capture and close all windows
cap.release()
out.release()  # Release the VideoWriter object
cv2.destroyAllWindows()



0: 384x640 1 car, 1 truck, 196.6ms
Speed: 6.8ms preprocess, 196.6ms inference, 1925.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 plate, 155.6ms
Speed: 1.0ms preprocess, 155.6ms inference, 7.0ms postprocess per image at shape (1, 3, 384, 640)
traindata.csv created successfully.

0: 384x640 1 car, 1 truck, 12.8ms
Speed: 5.7ms preprocess, 12.8ms inference, 2.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 plate, 15.0ms
Speed: 2.9ms preprocess, 15.0ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)
traindata.csv created successfully.

0: 384x640 1 car, 1 truck, 15.7ms
Speed: 15.3ms preprocess, 15.7ms inference, 3.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 plate, 13.5ms
Speed: 2.0ms preprocess, 13.5ms inference, 4.5ms postprocess per image at shape (1, 3, 384, 640)
traindata.csv created successfully.

0: 384x640 1 car, 1 truck, 14.3ms
Speed: 3.0ms preprocess, 14.3ms inference, 3.0ms postprocess per image at shap

In [ ]:
df